In [ ]:
"""Figure 4G. Nearest distances between nodes on different tracheoles."""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.stats import gaussian_kde

node_file = Path('data/catmaid_nodes.csv')
workers = 4
memory_mb = 256
k_start, k_max = 16, 256
n_boot = 10_000
seed = 20260911
thresholds_um = [2, 5, 10, 20]

nodes = pd.read_csv(node_file, usecols=['skeleton_id', 'x_nm', 'y_nm', 'z_nm'])
xyz = nodes[['x_nm', 'y_nm', 'z_nm']].to_numpy(float)
sids = nodes['skeleton_id'].to_numpy(np.int64)


def nearest_other_tracheole(xyz, sids):
    tree = cKDTree(xyz)
    distances_nm = np.empty(len(xyz), dtype=float)
    remaining = np.arange(len(xyz))
    k = k_start

    while len(remaining):
        batch_size = max(1, min(len(remaining), memory_mb * 1024**2 // (40 * k)))
        resolved = np.zeros(len(remaining), dtype=bool)

        for start in range(0, len(remaining), batch_size):
            ids = remaining[start:start + batch_size]
            distances, indices = tree.query(xyz[ids], k=k, workers=workers)
            other = sids[indices] != sids[ids, None]
            found = other.any(axis=1)
            resolved[start:start + len(ids)] = found

            if np.any(found):
                first = other[found].argmax(axis=1)
                distances_nm[ids[found]] = distances[found, first]

        remaining = remaining[~resolved]
        if k == k_max:
            break
        k = min(k * 2, k_max)

    for sid in np.unique(sids[remaining]):
        ids = remaining[sids[remaining] == sid]
        other = np.flatnonzero(sids != sid)
        distances_nm[ids] = cKDTree(xyz[other]).query(xyz[ids], workers=workers)[0]

    return distances_nm / 1000


distances_um = nearest_other_tracheole(xyz, sids)
q1, median, q3 = np.quantile(distances_um, [0.25, 0.5, 0.75])

# Per-tracheole summaries and bootstrap confidence intervals
node_stats = pd.DataFrame({'skeleton_id': sids, 'distance_um': distances_um})
per_tracheole = node_stats.groupby('skeleton_id')['distance_um'].agg(['count', 'mean', 'median'])
for threshold in thresholds_um:
    per_tracheole[f'fraction_within_{threshold}um'] = (
        node_stats.assign(within=node_stats['distance_um'] <= threshold)
        .groupby('skeleton_id')['within'].mean()
    )

rng = np.random.default_rng(seed)


def median_ci(values):
    values = np.asarray(values, dtype=float)
    bootstrap = np.array([
        np.median(rng.choice(values, size=len(values), replace=True))
        for _ in range(n_boot)
    ])
    return np.quantile(bootstrap, [0.025, 0.975])


mean_summary = per_tracheole['mean'].to_numpy()
median_summary = per_tracheole['median'].to_numpy()
mean_ci = median_ci(mean_summary)
median_ci_values = median_ci(median_summary)

print(f"Figure 4G: {len(per_tracheole)} tracheoles, {len(distances_um):,} CATMAID nodes")
print(f"Pooled median = {median:.3f} µm, IQR = {q1:.3f}–{q3:.3f} µm")
print(f"Pooled mean ± SD = {distances_um.mean():.3f} ± {distances_um.std(ddof=1):.3f} µm")
for threshold in thresholds_um:
    print(f"Nodes within {threshold} µm = {100 * np.mean(distances_um <= threshold):.1f}%")
print(f"Median per-tracheole mean = {np.median(mean_summary):.3f} µm, 95% CI = {mean_ci[0]:.3f}–{mean_ci[1]:.3f} µm")
print(f"Median per-tracheole median = {np.median(median_summary):.3f} µm, 95% CI = {median_ci_values[0]:.3f}–{median_ci_values[1]:.3f} µm")

# Figure 4G
plt.rcParams.update({
    'font.family': 'Arial', 'font.size': 11, 'axes.labelsize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'axes.linewidth': 0.8,
    'xtick.direction': 'out', 'ytick.direction': 'out', 'svg.fonttype': 'none'
})

fig, ax = plt.subplots(figsize=(4.4, 3.4))
bins = np.histogram_bin_edges(distances_um, bins='fd')
ax.hist(distances_um, bins=bins, density=True, color='#6F97B5',
        edgecolor='white', linewidth=0.45, alpha=0.72)
x = np.linspace(0, distances_um.max(), 500)
ax.plot(x, gaussian_kde(distances_um)(x), color='#244A66', lw=1.6)
ax.axvspan(q1, q3, color='#244A66', alpha=0.08, lw=0)
ax.axvline(median, color='#202020', lw=1.15, ls='--')
ax.text(0.98, 0.96, f'Median {median:.2f} µm\nIQR {q1:.2f}–{q3:.2f} µm',
        transform=ax.transAxes, ha='right', va='top', fontsize=10)
ax.set(xlabel='Distance to nearest tracheole (µm)', ylabel='Density', xlim=(0, None), ylim=(0, None))
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(length=3, width=0.8)
ax.margins(x=0)
fig.tight_layout()
fig.savefig('fig4g.svg', bbox_inches='tight')
plt.show()